# Faithfulness Metrics

Compare deletion, faithfulness correlation, comprehensiveness, and sufficiency under one explicit median-replacement protocol.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from xai_eval.data import breast_cancer_split
from xai_eval.models import make_models, positive_probability
from xai_eval.explainers import local_perturbation_attribution
from xai_eval.metrics import deletion_curve, deletion_auc, faithfulness_correlation, comprehensiveness, sufficiency

x_train, x_test, y_train, _, feature_names = breast_cancer_split(random_state=42)
baseline = np.median(x_train, axis=0)
model = make_models(random_state=42)["LogisticRegression"].fit(x_train, y_train)
x = x_test[0]
score_fn = lambda point: positive_probability(model, point)
attribution = local_perturbation_attribution(model, x, baseline=baseline)
selected = np.argsort(-np.abs(attribution))[:6]
fractions, scores = deletion_curve(score_fn, x, attribution, baseline=baseline, steps=20)
print({"deletion_auc": deletion_auc(score_fn, x, attribution, baseline=baseline),
       "faithfulness_correlation": faithfulness_correlation(score_fn, x, attribution, baseline=baseline),
       "comprehensiveness": comprehensiveness(score_fn, x, selected, baseline=baseline),
       "sufficiency": sufficiency(score_fn, x, selected, baseline=baseline)})
plt.plot(fractions, scores, marker="o"); plt.xlabel("Fraction removed"); plt.ylabel("Class 1 probability"); plt.show()

In [ ]:
# Replacing the same feature with zero versus the training median changes the question.
zero_curve = deletion_curve(score_fn, x, attribution, baseline=0.0, steps=20)
median_auc = deletion_auc(score_fn, x, attribution, baseline=baseline)
zero_auc = deletion_auc(score_fn, x, attribution, baseline=0.0)
assert np.isfinite([median_auc, zero_auc]).all()
print({"median_baseline_auc": median_auc, "zero_baseline_auc": zero_auc})